In [ ]:
# ── CELL 1: Installs and imports ──────────────────────────────────────────

!pip install -q --upgrade sentence-transformers hdbscan datasets bert-score
!pip install -q rouge-score scikit-learn matplotlib seaborn pandas scipy

!apt-get install -y -qq pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import re
import time
import json
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats as scipy_stats

import hdbscan
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from bert_score import score as bert_score_compute

import requests
import subprocess
from google.colab import drive

warnings.filterwarnings("ignore")

print("Cell 1: installs and imports complete")

Cell 1: installs and imports complete


In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────

drive.mount('/content/drive')

# ──────────────────────────────────────────────────────────────────────────
# PER-RUN KNOBS — change these two per Colab session, nothing else.
# ──────────────────────────────────────────────────────────────────────────

# Pick ONE model per session. Use the exact ollama tag (case-sensitive).
# Available options for this study:
#   "qwen2.5:3b"   ← Run 0 (synth=on) DONE, Run 1 (synth=off) next
#   "qwen2.5:7b"   ← Run 2: larger same family
#   "llama3.1:8b"  ← Run 3: different family, similar scale
# (Alternates if a model fails to load: "mistral:7b", "qwen2.5:14b")
PIPELINE_MODEL = "llama3.1:8b"

# Synthesis ablation flag. Flip to False for the synth-off run.
SYNTHESIZE = True

# ──────────────────────────────────────────────────────────────────────────
# RUN_TAG — auto-derived. Do not edit by hand; it defines the output folder.
# ──────────────────────────────────────────────────────────────────────────
RUN_TAG = f"{PIPELINE_MODEL.replace(':', '-')}_synth_{'on' if SYNTHESIZE else 'off'}"

BASE_PATH = "/content/drive/MyDrive/thesis_pipeline"
RUN_PATH  = f"{BASE_PATH}/runs/{RUN_TAG}"

# Per-run output paths (isolated per model + synth state)
OUTPUT_PATH            = f"{RUN_PATH}/outputs"
GRAPHS_PATH            = f"{OUTPUT_PATH}/graphs"
AGGREGATE_GRAPHS_PATH  = f"{GRAPHS_PATH}/aggregate"
PER_TOPIC_GRAPHS_PATH  = f"{GRAPHS_PATH}/per_topic"
PER_TOPIC_OUTPUTS_PATH = f"{OUTPUT_PATH}/per_topic_outputs"

# GLOBAL dataset cache — shared across ALL runs so every run evaluates the
# same 30 documents. Do not move this under RUN_PATH.
DATASET_CACHE_PATH     = f"{BASE_PATH}/outputs/dataset_cache"

for p in [RUN_PATH, OUTPUT_PATH, GRAPHS_PATH, AGGREGATE_GRAPHS_PATH,
          PER_TOPIC_GRAPHS_PATH, PER_TOPIC_OUTPUTS_PATH, DATASET_CACHE_PATH]:
    Path(p).mkdir(parents=True, exist_ok=True)

# Model + generation
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
TEMPERATURE     = 0.0
NUM_CTX         = 16384
NUM_PREDICT     = 4096

# Baseline behavior
LONG_INPUT_THRESHOLD = 20000
MAP_REDUCE_CHUNK     = 18000

# Refinement behavior — FROZEN across all runs for fair comparison
MIN_LOOPS            = 3
MAX_LOOPS            = 8
CLUSTER_CHAR_BUDGET  = 6000
PRIOR_ADDITIONS_CAP  = 30
QUOTE_OVERLAP_THRESHOLD = 0.75

# Clustering — FROZEN across all runs
HDBSCAN_MIN_SIZE  = 3
OVERLAP_THRESHOLD = 0.25
CONTEXT_WINDOW    = 2

# ── Prose dataset: GovReport (stratified by length) ────────────────────────
DATASET_NAME           = "ccdv/govreport-summarization"
DATASET_SPLIT          = "test"
DATASET_SAMPLE_SEED    = 42     # FROZEN — same sample across all runs
STRATIFIED_SAMPLE_SIZE = 30     # FROZEN — 10 short + 10 medium + 10 long

# Length buckets in characters (raw text, not summary)
SHORT_RANGE  = (15000, 30000)
MEDIUM_RANGE = (30000, 60000)
LONG_RANGE   = (60000, 120000)

# Uniform task instruction passed to baseline, refinement and synthesis.
# The document itself is supplied separately by each stage:
#   - baseline single_pass embeds raw_text in its SOURCE DOCUMENT section
#   - baseline map_reduce embeds chunks (not the whole doc) per call
#   - refinement only sees the current cluster, not the whole doc
#   - synthesis only sees the draft and verified additions, not the whole doc
GROUNDED_PROMPT_TEMPLATE = "Read the following government report carefully. Write a comprehensive summary based ONLY on the information contained in the document. Cover the main findings, recommendations, key data points, and conclusions. Do not add information from outside the document, and do not speculate beyond what the text states."

# ── Evaluation control ────────────────────────────────────────────────────
EVAL_NUM_TOPICS = 30   # how many docs from the stratified sample to evaluate (1..30)

assert EVAL_NUM_TOPICS <= STRATIFIED_SAMPLE_SIZE, \
    f"EVAL_NUM_TOPICS ({EVAL_NUM_TOPICS}) cannot exceed sample size ({STRATIFIED_SAMPLE_SIZE})"
assert 1 <= EVAL_NUM_TOPICS, "EVAL_NUM_TOPICS must be at least 1"

print("Cell 2: configuration loaded")
print(f"  RUN_TAG               : {RUN_TAG}")
print(f"  Run path              : {RUN_PATH}")
print(f"  Input mode            : prose")
print(f"  Pipeline model        : {PIPELINE_MODEL}")
print(f"  Embedding model       : {EMBEDDING_MODEL}")
print(f"  Loops                 : MIN={MIN_LOOPS}, MAX={MAX_LOOPS}")
print(f"  Cluster char budget   : {CLUSTER_CHAR_BUDGET}")
print(f"  Synthesis enabled     : {SYNTHESIZE}")
print(f"  Prose dataset         : {DATASET_NAME} ({DATASET_SPLIT} split)")
print(f"  Stratified sample     : {STRATIFIED_SAMPLE_SIZE} docs "
      f"({SHORT_RANGE[0]:,}-{SHORT_RANGE[1]:,} / "
      f"{MEDIUM_RANGE[0]:,}-{MEDIUM_RANGE[1]:,} / "
      f"{LONG_RANGE[0]:,}-{LONG_RANGE[1]:,} chars)")
print(f"  Sample seed           : {DATASET_SAMPLE_SEED} (reproducible)")
print(f"  Topics to evaluate    : {EVAL_NUM_TOPICS} of {STRATIFIED_SAMPLE_SIZE}")
print(f"  Dataset cache (global): {DATASET_CACHE_PATH}")
print(f"  Outputs (per-run)     : {OUTPUT_PATH}")

Mounted at /content/drive
Cell 2: configuration loaded
  RUN_TAG               : llama3.1-8b_synth_on
  Run path              : /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on
  Input mode            : prose
  Pipeline model        : llama3.1:8b
  Embedding model       : all-MiniLM-L6-v2
  Loops                 : MIN=3, MAX=8
  Cluster char budget   : 6000
  Synthesis enabled     : True
  Prose dataset         : ccdv/govreport-summarization (test split)
  Stratified sample     : 30 docs (15,000-30,000 / 30,000-60,000 / 60,000-120,000 chars)
  Sample seed           : 42 (reproducible)
  Topics to evaluate    : 30 of 30
  Dataset cache (global): /content/drive/MyDrive/thesis_pipeline/outputs/dataset_cache
  Outputs (per-run)     : /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/outputs


In [ ]:
# ── CELL 3: Ollama setup, model load, embedding model load ────────────────

print(f"=== Starting run: {RUN_TAG} ===")
print(f"  Model    : {PIPELINE_MODEL}")
print(f"  Synthesis: {SYNTHESIZE}")
print(f"  Saving to: {RUN_PATH}\n")

OLLAMA_LOG_PATH = f"{RUN_PATH}/ollama.log"
Path(RUN_PATH).mkdir(parents=True, exist_ok=True)


def _start_ollama_server():
    """Start (or restart) the Ollama subprocess with stderr captured to a log file."""
    subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
    time.sleep(2)
    log_f = open(OLLAMA_LOG_PATH, "ab")
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_f,
        stderr=log_f,
    )
    # Wait for the server to come up by polling the /api/tags endpoint
    for _ in range(20):
        time.sleep(1)
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                return proc
        except requests.exceptions.RequestException:
            continue
    raise RuntimeError(f"Ollama did not come up after 20s. Check {OLLAMA_LOG_PATH}")


ollama_process = _start_ollama_server()

print(f"Pulling pipeline model: {PIPELINE_MODEL}")
pull_result = subprocess.run(
    ["ollama", "pull", PIPELINE_MODEL],
    capture_output=True,
    text=True,
)
if pull_result.returncode != 0:
    print(f"  stderr: {pull_result.stderr[:500]}")
    raise RuntimeError(
        f"Ollama pull failed for '{PIPELINE_MODEL}'. "
        f"Check the model tag spelling in Cell 2 against the comments."
    )

ps_result = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print(f"\nOllama process status:")
print(ps_result.stdout)


def _ensure_ollama_alive():
    """If the Ollama subprocess died (OOM, kernel kill, etc.), restart it
    and re-pull the model. Returns True if a restart happened."""
    global ollama_process
    if ollama_process.poll() is None:
        # Subprocess is still running. Quick health ping to confirm responsive.
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=3)
            if r.status_code == 200:
                return False
        except requests.exceptions.RequestException:
            pass
    # Either subprocess died or server is unresponsive. Hard restart.
    print(f"\n  [!] Ollama is down. Restarting subprocess and reloading model...")
    ollama_process = _start_ollama_server()
    # Re-warm the model into memory (pull is a no-op if it's already cached)
    subprocess.run(["ollama", "pull", PIPELINE_MODEL], capture_output=True, text=True)
    print(f"  [+] Ollama restarted and {PIPELINE_MODEL} reloaded.\n")
    return True


def llm_call(prompt, num_predict=None, max_retries=4):
    """
    Call Ollama with automatic recovery.
    - 500/timeout/connection errors trigger restart + retry
    - Model is kept loaded forever via keep_alive=-1 (prevents idle unload)
    - 4xx errors (bad request) are not retried
    """
    payload = {
        "model": PIPELINE_MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": -1,
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": NUM_CTX,
            "num_predict": num_predict if num_predict is not None else NUM_PREDICT,
        },
    }

    last_error = None
    for attempt in range(max_retries + 1):
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json=payload,
                timeout=600,
            )
            response.raise_for_status()
            return response.json()["response"]
        except (requests.exceptions.HTTPError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.ChunkedEncodingError) as e:
            last_error = e
            status = getattr(getattr(e, "response", None), "status_code", None)
            # 4xx (bad request, model not found) are not retryable
            if status is not None and 400 <= status < 500:
                raise

            if attempt < max_retries:
                # Restart Ollama if subprocess is dead or unresponsive
                _ensure_ollama_alive()
                wait = 2 ** attempt  # 1, 2, 4, 8 seconds
                print(f"      (Ollama {type(e).__name__}"
                      f"{' ' + str(status) if status else ''}, "
                      f"retry {attempt + 1}/{max_retries} in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last_error


print(f"\nLoading embedding model: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")
print(f"  Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

test_response = llm_call("Reply with the single word: ready", num_predict=10)
print(f"\nLLM smoke test: {test_response.strip()[:50]}")

print(f"\nCell 3: Ollama and embedding model ready")
print(f"  Ollama log: {OLLAMA_LOG_PATH}")
print(f"  Auto-restart enabled on subprocess death or server unresponsiveness")
print(f"  keep_alive=-1 (model stays loaded)")

=== Starting run: llama3.1-8b_synth_on ===
  Model    : llama3.1:8b
  Synthesis: True
  Saving to: /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on

Pulling pipeline model: llama3.1:8b

Ollama process status:
NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Embedding dimension: 384

LLM smoke test: Ready

Cell 3: Ollama and embedding model ready
  Ollama log: /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/ollama.log
  Auto-restart enabled on subprocess death or server unresponsiveness
  keep_alive=-1 (model stays loaded)


In [ ]:
# ── CELL 5: Loaders as functions ──────────────────────────────────────────

# Module-level cache so the stratified sample is loaded from disk (or built)
# exactly once per Python session, not once per topic.
_GOVREPORT_SAMPLE_CACHE = None


def build_govreport_sample(
    dataset_name=DATASET_NAME,
    split=DATASET_SPLIT,
    sample_size=STRATIFIED_SAMPLE_SIZE,
    seed=DATASET_SAMPLE_SEED,
    short_range=SHORT_RANGE,
    medium_range=MEDIUM_RANGE,
    long_range=LONG_RANGE,
    cache_path=DATASET_CACHE_PATH,
    force_rebuild=False,
):
    """
    Build a length-stratified sample of GovReport documents and cache it to Drive.

    Returns a list of dicts: [{doc_id, raw_text, reference_summary, length_bucket}, ...].
    Stratification: equal counts from short/medium/long character-length buckets.

    The sample is deterministic given (seed, sample_size, range definitions). Once
    cached, subsequent calls (same or different Colab session) return the same docs
    without re-downloading the dataset.
    """
    cache_file = f"{cache_path}/govreport_sample_s{seed}_n{sample_size}.pkl"

    if not force_rebuild and Path(cache_file).exists():
        with open(cache_file, "rb") as f:
            sample = pickle.load(f)
        print(f"  Loaded cached sample from {cache_file}")
        print(f"    {len(sample)} documents")
        return sample

    print(f"  No cache found. Downloading {dataset_name} ({split} split)...")
    print(f"  First run will take 2-5 minutes and ~1GB disk.")
    ds = load_dataset(dataset_name, split=split)

    text_col = "report" if "report" in ds.column_names else "article"
    summary_col = "summary" if "summary" in ds.column_names else "abstract"
    print(f"  Dataset loaded: {len(ds):,} documents")
    print(f"  Using columns: text='{text_col}', summary='{summary_col}'")

    per_bucket = sample_size // 3
    if sample_size % 3 != 0:
        raise ValueError(f"sample_size ({sample_size}) must be divisible by 3 for stratification")

    buckets = {"short": [], "medium": [], "long": []}
    bucket_ranges = {
        "short":  short_range,
        "medium": medium_range,
        "long":   long_range,
    }

    print(f"  Scanning documents and bucketing by length...")
    for idx, row in enumerate(ds):
        text = row[text_col]
        n = len(text)
        for bucket_name, (lo, hi) in bucket_ranges.items():
            if lo <= n < hi:
                buckets[bucket_name].append({
                    "dataset_idx": idx,
                    "doc_id": f"govreport_{idx:05d}",
                    "raw_text": text,
                    "reference_summary": row[summary_col],
                    "length_bucket": bucket_name,
                    "char_count": n,
                })
                break

    print(f"  Found per bucket: "
          f"short={len(buckets['short'])}, "
          f"medium={len(buckets['medium'])}, "
          f"long={len(buckets['long'])}")

    for bucket_name in ("short", "medium", "long"):
        if len(buckets[bucket_name]) < per_bucket:
            raise ValueError(
                f"Bucket '{bucket_name}' has only {len(buckets[bucket_name])} "
                f"documents, need {per_bucket}. Widen the range in Cell 2."
            )

    rng = np.random.default_rng(seed)
    sample = []
    for bucket_name in ("short", "medium", "long"):
        pool = buckets[bucket_name]
        chosen_indices = rng.choice(len(pool), size=per_bucket, replace=False)
        sample.extend([pool[i] for i in chosen_indices])

    Path(cache_path).mkdir(parents=True, exist_ok=True)
    with open(cache_file, "wb") as f:
        pickle.dump(sample, f)
    print(f"  Cached sample to {cache_file}")
    print(f"  Sample size: {len(sample)} ({per_bucket} per bucket)")

    return sample


def get_govreport_sample():
    """Lazy session-cache wrapper around build_govreport_sample()."""
    global _GOVREPORT_SAMPLE_CACHE
    if _GOVREPORT_SAMPLE_CACHE is None:
        _GOVREPORT_SAMPLE_CACHE = build_govreport_sample()
    return _GOVREPORT_SAMPLE_CACHE


def load_prose(doc_id):
    """
    Load one GovReport document by doc_id (e.g. 'govreport_03192').

    Returns {raw_text, units, source_title, reference_summary, length_bucket}.
    Downstream cells already consume the first three fields; reference_summary
    is used by BERTScore in Cell 11.
    """
    sample = get_govreport_sample()
    match = next((d for d in sample if d["doc_id"] == doc_id), None)
    if match is None:
        available = [d["doc_id"] for d in sample][:5]
        raise ValueError(
            f"doc_id '{doc_id}' not found in cached sample. "
            f"First 5 available: {available}..."
        )

    raw_text = match["raw_text"]

    sentences = re.split(r"(?<=[.?!])\s+(?=[A-Z])", raw_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 40]

    return {
        "raw_text": raw_text,
        "units": sentences,
        "source_title": doc_id,
        "reference_summary": match["reference_summary"],
        "length_bucket": match["length_bucket"],
    }


print("Cell 5: loader functions defined")
print(f"  build_govreport_sample() : downloads + stratifies + caches")
print(f"  get_govreport_sample()   : session-cached accessor")
print(f"  load_prose(doc_id)       : loads one doc by ID from cached sample")

Cell 5: loader functions defined
  build_govreport_sample() : downloads + stratifies + caches
  get_govreport_sample()   : session-cached accessor
  load_prose(doc_id)       : loads one doc by ID from cached sample
  load_code(filename)      : unchanged from before


In [ ]:
# ── CELL 4: Sample preview helper ─────────────────────────────────────────

def preview_topics(num_topics=None, preview_chars_each_side=600):
    """
    Preview the stratified GovReport sample. The first call triggers the
    one-time dataset download and cache build; subsequent calls are instant.
    """
    n = num_topics if num_topics is not None else EVAL_NUM_TOPICS
    n = min(n, STRATIFIED_SAMPLE_SIZE)

    try:
        sample = get_govreport_sample()
    except Exception as e:
        print(f"FATAL: failed to build GovReport sample.")
        print(f"  {type(e).__name__}: {e}")
        return

    print(f"\nPreviewing first {n} documents from stratified sample")
    print(f"  Total cached sample size: {len(sample)}")
    print(f"  Buckets: "
          f"short={sum(1 for d in sample if d['length_bucket']=='short')}, "
          f"medium={sum(1 for d in sample if d['length_bucket']=='medium')}, "
          f"long={sum(1 for d in sample if d['length_bucket']=='long')}")
    print("=" * 70)

    for i, doc in enumerate(sample[:n]):
        doc_id   = doc["doc_id"]
        bucket   = doc["length_bucket"]
        n_chars  = doc["char_count"]
        ref_sum  = doc["reference_summary"]
        content  = doc["raw_text"]

        print(f"\n[{i+1}/{n}]  doc_id: {doc_id}  bucket: {bucket}  chars: {n_chars:,}")
        print(f"        reference summary length: {len(ref_sum):,} chars")

        if n_chars > 2 * preview_chars_each_side:
            preview = (
                content[:preview_chars_each_side].strip()
                + "\n        --- ... (truncated) ... ---\n"
                + content[-preview_chars_each_side:].strip()
            )
        else:
            preview = content.strip()

        print(f"\n        --- first {preview_chars_each_side} chars and last {preview_chars_each_side} chars ---")
        print(preview)
        print(f"        --- end document preview ---")
        print("=" * 70)

    print(f"\nDone. Sample is cached. The evaluation prompt for every document is:")
    print(f"\n{GROUNDED_PROMPT_TEMPLATE[:300]}...\n")
    print(f"(Defined in Cell 2 as GROUNDED_PROMPT_TEMPLATE.)")


preview_topics()

  Loaded cached sample from /content/drive/MyDrive/thesis_pipeline/outputs/dataset_cache/govreport_sample_s42_n30.pkl
    30 documents

Previewing first 30 documents from stratified sample
  Total cached sample size: 30
  Buckets: short=10, medium=10, long=10

[1/30]  doc_id: govreport_00957  bucket: short  chars: 19,129
        reference summary length: 4,589 chars

        --- first 600 chars and last 600 chars ---
Since 2009, the federal government has been shifting its data storage needs to cloud-based services and away from agency-owned, in-house data centers. This shift is intended to achieve two goals: reduce the total investment by the federal government in information technology (IT), which currently stands at about $90 billion each year, and realize other stated advantages of cloud adoption , including efficiency, accessibility, collaboration, reliability, and security. However, challenges remain as agencies shift to cloud services. According to a survey conducted in Septembe

In [ ]:
# ── CELL 6: Clustering as functions ───────────────────────────────────────

def embed_units(units, context_window=CONTEXT_WINDOW):
    if not units:
        return np.zeros((0, 384))

    contextualized = []
    for i, unit in enumerate(units):
        start = max(0, i - context_window)
        end = min(len(units), i + context_window + 1)
        context_block = " ".join(units[start:end])
        contextualized.append(context_block)

    embeddings = embedding_model.encode(
        contextualized,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return embeddings


def _split_largest_cluster(embeddings, labels):
    """Split the largest existing cluster into 2 via KMeans. Returns updated labels."""
    cluster_sizes = {}
    for cid in set(labels):
        if cid == -1:
            continue
        cluster_sizes[cid] = int(np.sum(labels == cid))

    if not cluster_sizes:
        # Edge case: only noise. Fall back to splitting all points.
        target_indices = np.arange(len(labels))
        existing_max = -1
    else:
        largest_cid = max(cluster_sizes, key=cluster_sizes.get)
        target_indices = np.where(labels == largest_cid)[0]
        existing_max = max(cluster_sizes.keys())

    if len(target_indices) < 2:
        return labels  # cannot split

    sub_emb = embeddings[target_indices]
    km = KMeans(n_clusters=2, n_init=10, random_state=42)
    sub_labels = km.fit_predict(sub_emb)

    new_cid = existing_max + 1
    for local_idx, point_idx in enumerate(target_indices):
        if sub_labels[local_idx] == 1:
            labels[point_idx] = new_cid
        # sub_labels==0 keeps the original (or stays as -1 in fallback case;
        # we relabel below)
    if existing_max == -1:
        # fallback path: half got new_cid, the other half is still -1
        for local_idx, point_idx in enumerate(target_indices):
            if sub_labels[local_idx] == 0:
                labels[point_idx] = 0
    return labels


def _merge_closest_clusters(embeddings, labels):
    """Merge the smallest cluster into its nearest neighbor cluster."""
    cluster_sizes = {}
    for cid in set(labels):
        if cid == -1:
            continue
        cluster_sizes[cid] = int(np.sum(labels == cid))

    if len(cluster_sizes) < 2:
        return labels

    smallest_cid = min(cluster_sizes, key=cluster_sizes.get)
    smallest_indices = np.where(labels == smallest_cid)[0]
    smallest_centroid = embeddings[smallest_indices].mean(axis=0)

    other_cids = [c for c in cluster_sizes if c != smallest_cid]
    best_cid = None
    best_dist = float("inf")
    for cid in other_cids:
        cid_indices = np.where(labels == cid)[0]
        cid_centroid = embeddings[cid_indices].mean(axis=0)
        dist = np.linalg.norm(smallest_centroid - cid_centroid)
        if dist < best_dist:
            best_dist = dist
            best_cid = cid

    labels[labels == smallest_cid] = best_cid
    return labels


def enforce_cluster_bounds(embeddings, min_loops=MIN_LOOPS, max_loops=MAX_LOOPS,
                           starting_min_size=HDBSCAN_MIN_SIZE):
    """
    HDBSCAN runs once with its natural parameters. Then we coerce the cluster
    count into [min_loops, max_loops]:
      - too few clusters  -> split the largest cluster via 2-means
      - too many clusters -> merge the smallest into its nearest neighbor
    This guarantees the recursion loop has between min_loops and max_loops
    clusters to iterate over, regardless of what HDBSCAN naturally finds.
    """
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=starting_min_size,
        prediction_data=True,
    )
    labels = clusterer.fit_predict(embeddings).copy()

    raw_clusters = len(set(labels) - {-1})
    noise_points = int((labels == -1).sum())

    def n_real_clusters(lbls):
        return len(set(lbls) - {-1})

    # Safety: if HDBSCAN returned ALL noise, seed one cluster from all points
    if n_real_clusters(labels) == 0:
        labels[:] = 0

    # Split until we have at least min_loops
    safety = 0
    while n_real_clusters(labels) < min_loops and safety < 50:
        labels = _split_largest_cluster(embeddings, labels)
        safety += 1

    # Merge until we have at most max_loops.
    # Budget scales with cluster count (each merge removes exactly one), so a
    # heavily fragmented document can still be coerced down to max_loops.
    merge_budget = n_real_clusters(labels) + 5
    safety = 0
    while n_real_clusters(labels) > max_loops and safety < merge_budget:
        before = n_real_clusters(labels)
        labels = _merge_closest_clusters(embeddings, labels)
        safety += 1
        if n_real_clusters(labels) >= before:   # no progress, bail safely
            break

    # Compact label space to 0..K-1
    unique_labels = sorted(set(labels) - {-1})
    relabel_map = {old: new for new, old in enumerate(unique_labels)}
    relabeled = np.array([relabel_map[l] if l != -1 else -1 for l in labels])

    diag = {"raw_clusters_hdbscan": raw_clusters, "noise_points": noise_points}
    return clusterer, relabeled, len(unique_labels), diag


def reassign_noise_to_nearest_cluster(embeddings, labels):
    if -1 not in labels:
        return labels

    n_clusters = len(set(labels) - {-1})
    centroids = {}
    for cid in range(n_clusters):
        cid_indices = np.where(labels == cid)[0]
        if len(cid_indices) > 0:
            centroids[cid] = embeddings[cid_indices].mean(axis=0)

    if not centroids:
        # Pathological: no real clusters at all. Put everything in cluster 0.
        labels[:] = 0
        return labels

    noise_indices = np.where(labels == -1)[0]
    for idx in noise_indices:
        emb = embeddings[idx]
        best_cid = None
        best_dist = float("inf")
        for cid, centroid in centroids.items():
            dist = np.linalg.norm(emb - centroid)
            if dist < best_dist:
                best_dist = dist
                best_cid = cid
        labels[idx] = best_cid if best_cid is not None else 0

    return labels


def build_cluster_map(units, embeddings, labels, clusterer,
                     overlap_threshold=OVERLAP_THRESHOLD):
    n_clusters = len(set(labels) - {-1})

    # Soft-membership only works when HDBSCAN itself produced >=1 real cluster
    # AND the labels we have weren't reshaped by manual split/merge. Try, but
    # fail open: if the shape doesn't match, skip soft-membership.
    soft_membership = None
    try:
        sm = hdbscan.all_points_membership_vectors(clusterer)
        if sm is not None and sm.ndim == 2 and sm.shape[1] == n_clusters:
            soft_membership = sm
    except Exception:
        soft_membership = None

    cluster_map = []
    for i, unit in enumerate(units):
        primary = int(labels[i])
        tags = [f"c{primary}"]

        if soft_membership is not None:
            for cid in range(n_clusters):
                if cid == primary:
                    continue
                if soft_membership[i, cid] >= overlap_threshold:
                    tags.append(f"c{cid}")

        tag_str = "|".join(tags)
        cluster_map.append(f"{unit} | {tag_str}")

    return cluster_map


def compute_centroids(embeddings, labels, n_clusters):
    centroids = {}
    for cid in range(n_clusters):
        cid_indices = np.where(labels == cid)[0]
        if len(cid_indices) > 0:
            centroids[cid] = embeddings[cid_indices].mean(axis=0)
    return centroids


def cluster_input(units):
    if len(units) < 2:
        raise ValueError(f"Cannot cluster {len(units)} units. Need at least 2.")

    embeddings = embed_units(units)
    clusterer, labels, n_clusters, cluster_diag = enforce_cluster_bounds(embeddings)
    labels = reassign_noise_to_nearest_cluster(embeddings, labels)
    # Recompute n_clusters in case noise reassignment or merges changed counts
    n_clusters = len(set(labels) - {-1})
    cluster_map = build_cluster_map(units, embeddings, labels, clusterer)
    centroids = compute_centroids(embeddings, labels, n_clusters)

    return {
        "embeddings": embeddings,
        "labels": labels,
        "n_clusters": n_clusters,
        "raw_clusters_hdbscan": cluster_diag["raw_clusters_hdbscan"],
        "noise_points": cluster_diag["noise_points"],
        "cluster_map": cluster_map,
        "centroids": centroids,
        "clusterer": clusterer,
    }

print("Cell 6: clustering functions defined")
print(f"  Pipeline: embed_units → enforce_cluster_bounds (split/merge to [MIN_LOOPS, MAX_LOOPS]) → reassign_noise → build_cluster_map → compute_centroids")
print(f"  Bounds:   n_clusters coerced into [{MIN_LOOPS}, {MAX_LOOPS}] regardless of HDBSCAN's natural output")


Cell 6: clustering functions defined
  Pipeline: embed_units → enforce_cluster_bounds (split/merge to [MIN_LOOPS, MAX_LOOPS]) → reassign_noise → build_cluster_map → compute_centroids
  Bounds:   n_clusters coerced into [3, 8] regardless of HDBSCAN's natural output


In [ ]:
# ── CELL 7: Baseline as a function ────────────────────────────────────────

def generate_baseline_single_pass(raw_text, prompt):
    full_prompt = f"""TASK: {prompt}

SOURCE DOCUMENT:
{raw_text}

Write a complete answer to the task using only information from the source document above."""

    t0 = time.time()
    response = llm_call(full_prompt)
    elapsed = time.time() - t0

    return {
        "draft": response.strip(),
        "method": "single_pass",
        "calls": 1,
        "time": elapsed,
        "chunks_processed": 1,
    }


def generate_baseline_map_reduce(raw_text, prompt, chunk_size=MAP_REDUCE_CHUNK):
    chunks = []
    pos = 0
    while pos < len(raw_text):
        end = min(pos + chunk_size, len(raw_text))
        if end < len(raw_text):
            last_period = raw_text.rfind(". ", pos, end)
            if last_period > pos + chunk_size // 2:
                end = last_period + 2
        chunks.append(raw_text[pos:end])
        pos = end

    chunk_summaries = []
    total_calls = 0
    t0 = time.time()

    for i, chunk in enumerate(chunks):
        map_prompt = f"""TASK: {prompt}

SOURCE DOCUMENT (part {i+1} of {len(chunks)}):
{chunk}

Extract from this part of the document any information relevant to the task. Be concise."""
        response = llm_call(map_prompt)
        chunk_summaries.append(response.strip())
        total_calls += 1

    combined = "\n\n".join(
        f"[from part {i+1}]\n{summary}"
        for i, summary in enumerate(chunk_summaries)
    )

    reduce_prompt = f"""TASK: {prompt}

PARTIAL ANSWERS FROM EACH PART OF THE SOURCE DOCUMENT:
{combined}

Combine the partial answers above into one coherent, complete answer to the task. Remove duplicates, resolve any contradictions, and produce a single unified response."""

    final_response = llm_call(reduce_prompt)
    total_calls += 1
    elapsed = time.time() - t0

    return {
        "draft": final_response.strip(),
        "method": "map_reduce",
        "calls": total_calls,
        "time": elapsed,
        "chunks_processed": len(chunks),
    }


def generate_baseline(raw_text, prompt, long_threshold=LONG_INPUT_THRESHOLD):
    if len(raw_text) < long_threshold:
        return generate_baseline_single_pass(raw_text, prompt)
    else:
        return generate_baseline_map_reduce(raw_text, prompt)


print("Cell 7: baseline functions defined")
print(f"  Single-pass for inputs < {LONG_INPUT_THRESHOLD:,} chars")
print(f"  Map-reduce  for inputs >= {LONG_INPUT_THRESHOLD:,} chars (chunk size: {MAP_REDUCE_CHUNK:,})")

Cell 7: baseline functions defined
  Single-pass for inputs < 20,000 chars
  Map-reduce  for inputs >= 20,000 chars (chunk size: 18,000)


In [ ]:
# ── CELL 8: Refinement as a function ──────────────────────────────────────

def get_cluster_sentences(cluster_map, cluster_id):
    sentences = []
    for entry in cluster_map:
        text, tag = entry.rsplit(" | ", 1)
        cluster_ids = tag.split("|")
        if f"c{cluster_id}" in cluster_ids:
            sentences.append(text.strip())
    return sentences

def truncate_cluster_if_too_big(cluster_id, cluster_sentences, embeddings, cluster_map,
                                 char_budget=CLUSTER_CHAR_BUDGET):
    joined_size = sum(len(s) for s in cluster_sentences) + len(cluster_sentences)
    if joined_size <= char_budget:
        return cluster_sentences, False

    member_indices = [
        i for i, entry in enumerate(cluster_map)
        if f"c{cluster_id}" in entry.rsplit(" | ", 1)[1].split("|")
    ]
    centroid = embeddings[member_indices].mean(axis=0)

    ranked = sorted(
        member_indices,
        key=lambda i: np.linalg.norm(embeddings[i] - centroid),
    )

    kept_indices = set()
    running_size = 0
    for i in ranked:
        text = cluster_map[i].rsplit(" | ", 1)[0].strip()
        addition = len(text) + 1
        if running_size + addition > char_budget:
            break
        kept_indices.add(i)
        running_size += addition

    truncated = [
        cluster_map[i].rsplit(" | ", 1)[0].strip()
        for i in member_indices
        if i in kept_indices
    ]
    return truncated, True

def fuzzy_quote_match(quote, cluster_corpus, threshold=QUOTE_OVERLAP_THRESHOLD):
    quote_clean = quote.lower().strip()
    if not quote_clean:
        return False

    if quote_clean in cluster_corpus.lower():
        return True

    quote_tokens = set(re.findall(r"\w+", quote_clean))
    if not quote_tokens:
        return False

    corpus_tokens = set(re.findall(r"\w+", cluster_corpus.lower()))
    overlap = len(quote_tokens & corpus_tokens) / len(quote_tokens)

    return overlap >= threshold

def build_refine_prompt(cluster_sentences, current_output, prior_additions, original_task,
                         additions_cap=PRIOR_ADDITIONS_CAP):
    cluster_text = "\n".join(f"- {s}" for s in cluster_sentences)

    if prior_additions:
        capped = prior_additions[-additions_cap:] if len(prior_additions) > additions_cap else prior_additions
        prior_block = "\n".join(f"- {a}" for a in capped)
        if len(prior_additions) > additions_cap:
            prior_block = f"(showing last {additions_cap} of {len(prior_additions)} total)\n" + prior_block
    else:
        prior_block = "(none yet — this is the first refinement loop)"

    return f"""ORIGINAL TASK (the user's question):
{original_task}

CURRENT DRAFT (your answer to the task so far):
{current_output}

POINTS YOU HAVE ALREADY ADDED IN PRIOR LOOPS (do NOT repeat any of these):
{prior_block}

ONE TOPIC FROM THE SOURCE DOCUMENT (cluster of sentences from the source on one topic):
{cluster_text}

Your job: identify content in the TOPIC sentences above that is relevant to the ORIGINAL TASK but missing from or wrong in the CURRENT DRAFT.

Respond in EXACTLY one of these two formats:

If nothing in the topic is missing or wrong, output exactly:
NONE

Otherwise, output one or more lines:
ADD: <new point relevant to the task> || <verbatim quote from the topic sentences proving this point is in the source>
FIX: <something the draft says> -> <what it should say instead> || <verbatim quote from the topic sentences supporting the fix>

CRITICAL RULES:
- Every ADD and FIX must include a quote from the topic sentences (after `||`).
- Do NOT add content from your own knowledge. Only add what is supported by the topic sentences.
- Do NOT repeat anything in the CURRENT DRAFT or POINTS YOU HAVE ALREADY ADDED list.
- Output only NONE, ADD: lines, or FIX: lines. No other text.

Begin.
"""

def apply_refinement(response, current_output, prior_additions, cluster_sentences):
    response_stripped = response.strip()

    if not response_stripped:
        return current_output, prior_additions, "MALFORMED", {"additions": [], "fixes": [],
                                                                "rejected_ungrounded": [], "rejected_duplicates": []}

    upper = response_stripped.upper()
    if upper == "NONE" or upper.startswith("NONE\n") or upper.startswith("NONE "):
        return current_output, prior_additions, "UNCHANGED", {"additions": [], "fixes": [],
                                                                "rejected_ungrounded": [], "rejected_duplicates": []}

    cluster_corpus = " ".join(cluster_sentences)

    raw_additions = []
    raw_fixes = []
    rejected_ungrounded = []

    for line in response_stripped.splitlines():
        line = line.strip()
        if line.upper().startswith("ADD:"):
            content = line[4:].strip()
            if "||" not in content:
                rejected_ungrounded.append(("ADD without quote", content[:80]))
                continue
            point, quote = content.split("||", 1)
            point = point.strip()
            quote = quote.strip().strip('"').strip("'")
            if not fuzzy_quote_match(quote, cluster_corpus):
                rejected_ungrounded.append(("ADD with bad quote", point[:80]))
                continue
            if point:
                raw_additions.append(point)

        elif line.upper().startswith("FIX:") and "->" in line and "||" in line:
            content = line[4:].strip()
            fix_part, quote = content.rsplit("||", 1)
            quote = quote.strip().strip('"').strip("'")
            if not fuzzy_quote_match(quote, cluster_corpus):
                rejected_ungrounded.append(("FIX with bad quote", fix_part[:80]))
                continue
            if "->" in fix_part:
                old, new = fix_part.split("->", 1)
                old = old.strip()
                new = new.strip()
                if old and new:
                    raw_fixes.append((old, new))

    if not raw_additions and not raw_fixes and not rejected_ungrounded:
        return current_output, prior_additions, "MALFORMED", {"additions": [], "fixes": [],
                                                                "rejected_ungrounded": [], "rejected_duplicates": []}

    prior_set = {a.lower() for a in prior_additions}
    additions = []
    rejected_duplicates = []
    for a in raw_additions:
        if a.lower() in prior_set:
            rejected_duplicates.append(a)
        else:
            additions.append(a)
            prior_set.add(a.lower())

    new_output = current_output
    applied_fixes = []
    for old, new in raw_fixes:
        if old in new_output:
            new_output = new_output.replace(old, new, 1)
            applied_fixes.append((old, new))

    if additions:
        new_output = new_output.rstrip() + "\n\n" + "\n".join(f"- {a}" for a in additions)

    updated_prior = prior_additions + additions
    metadata = {
        "additions": additions,
        "fixes": applied_fixes,
        "rejected_ungrounded": rejected_ungrounded,
        "rejected_duplicates": rejected_duplicates,
    }

    if not additions and not applied_fixes:
        return current_output, prior_additions, "UNCHANGED", metadata

    return new_output, updated_prior, "REVISED", metadata

def pick_next_cluster(active_clusters, centroids, current_output):
    out_emb = embedding_model.encode([current_output], convert_to_numpy=True)[0]
    distances = {}
    for cid in active_clusters:
        c = centroids[cid]
        cos_sim = np.dot(out_emb, c) / (np.linalg.norm(out_emb) * np.linalg.norm(c) + 1e-9)
        distances[cid] = 1 - cos_sim
    return max(distances, key=distances.get)

def refine(draft, cluster_data, original_task, max_loops=MAX_LOOPS):
    cluster_map = cluster_data["cluster_map"]
    embeddings = cluster_data["embeddings"]
    centroids = cluster_data["centroids"]
    n_clusters = cluster_data["n_clusters"]

    current_output = draft
    all_additions = []
    active_clusters = set(range(n_clusters))
    processed_order = []
    loop_metadata = {}
    refinement_calls = 0
    refinement_time = 0.0

    for loop_num in range(1, max_loops + 1):
        if not active_clusters:
            break

        cluster_id = pick_next_cluster(active_clusters, centroids, current_output)
        full_cluster = get_cluster_sentences(cluster_map, cluster_id)
        cluster_sentences, was_truncated = truncate_cluster_if_too_big(
            cluster_id, full_cluster, embeddings, cluster_map
        )

        if not cluster_sentences:
            active_clusters.discard(cluster_id)
            continue

        refine_prompt = build_refine_prompt(cluster_sentences, current_output,
                                              all_additions, original_task)

        t0 = time.time()
        response = llm_call(refine_prompt)
        elapsed = time.time() - t0
        refinement_calls += 1
        refinement_time += elapsed

        new_output, all_additions, outcome, metadata = apply_refinement(
            response, current_output, all_additions, cluster_sentences
        )

        if outcome == "REVISED":
            current_output = new_output

        active_clusters.discard(cluster_id)
        processed_order.append(cluster_id)

        loop_metadata[loop_num] = {
            "cluster_id": int(cluster_id),
            "outcome": outcome,
            "active_remaining": len(active_clusters),
            "raw_response_chars": len(response),
            "loop_time": elapsed,
            "cluster_truncated": was_truncated,
            "cluster_size_full": len(full_cluster),
            "cluster_size_used": len(cluster_sentences),
            "additions_this_loop": len(metadata["additions"]),
            "fixes_this_loop": len(metadata["fixes"]),
            "rejected_ungrounded_this_loop": len(metadata["rejected_ungrounded"]),
            "rejected_duplicates_this_loop": len(metadata["rejected_duplicates"]),
            "additions_total_so_far": len(all_additions),
        }

    return {
        "final_output_raw": current_output,
        "all_additions": all_additions,
        "processed_order": processed_order,
        "loop_metadata": loop_metadata,
        "refinement_calls": refinement_calls,
        "refinement_time": refinement_time,
        "n_clusters_total": n_clusters,
        "n_clusters_processed": len(processed_order),
    }

print("Cell 8: refinement functions defined")
print(f"  Per-cluster char budget : {CLUSTER_CHAR_BUDGET}")
print(f"  Prior additions cap     : last {PRIOR_ADDITIONS_CAP} shown to model")
print(f"  Quote overlap threshold : {QUOTE_OVERLAP_THRESHOLD} (fuzzy match)")
print(f"  Max refinement loops    : {MAX_LOOPS}")

Cell 8: refinement functions defined
  Per-cluster char budget : 6000
  Prior additions cap     : last 30 shown to model
  Quote overlap threshold : 0.75 (fuzzy match)
  Max refinement loops    : 8


In [ ]:
# ── CELL 9: Synthesis as a function ───────────────────────────────────────

def build_synthesis_prompt(draft, additions, original_task):
    if not additions:
        return None

    additions_block = "\n".join(f"- {a}" for a in additions)

    return f"""ORIGINAL TASK (the user's question):
{original_task}

DRAFT ANSWER:
{draft}

ADDITIONAL VERIFIED POINTS (these were extracted from the source and validated; they must all appear in the final answer):
{additions_block}

Your job: produce a single, coherent answer to the original task that incorporates BOTH the draft answer AND every additional verified point above. Integrate the additional points into the prose naturally — do not list them at the end as bullets.

CRITICAL RULES:
- Every additional verified point above MUST appear in the final answer (rephrasing is allowed; deletion is NOT).
- Do NOT add any new information not present in the draft or the additional points.
- Do NOT remove any factual content from the draft.
- Output ONLY the final integrated answer. No preamble, no labels, no commentary.

Begin.
"""

def synthesize(draft, additions, original_task, enabled=SYNTHESIZE):
    if not enabled:
        return {
            "synthesized_output": draft,
            "synthesis_call_made": False,
            "synthesis_time": 0.0,
            "synthesis_skipped_reason": "SYNTHESIZE flag is False in Cell 2",
        }

    if not additions:
        return {
            "synthesized_output": draft,
            "synthesis_call_made": False,
            "synthesis_time": 0.0,
            "synthesis_skipped_reason": "no additions to integrate",
        }

    synthesis_prompt = build_synthesis_prompt(draft, additions, original_task)

    t0 = time.time()
    response = llm_call(synthesis_prompt)
    elapsed = time.time() - t0

    synthesized = response.strip()

    if len(synthesized) < len(draft) * 0.2:
        return {
            "synthesized_output": draft,
            "synthesis_call_made": True,
            "synthesis_time": elapsed,
            "synthesis_skipped_reason": f"output too short ({len(synthesized)} chars vs draft {len(draft)}), kept draft",
            "synthesis_raw_response": synthesized,
        }

    return {
        "synthesized_output": synthesized,
        "synthesis_call_made": True,
        "synthesis_time": elapsed,
        "synthesis_skipped_reason": None,
    }

print("Cell 9: synthesis function defined")
print(f"  SYNTHESIZE flag: {SYNTHESIZE}")
print(f"  Behavior: when enabled, runs one LLM call to integrate additions into prose")
print(f"  Safety: falls back to draft if synthesized output is < 20% of draft length")

Cell 9: synthesis function defined
  SYNTHESIZE flag: True
  Behavior: when enabled, runs one LLM call to integrate additions into prose
  Safety: falls back to draft if synthesized output is < 50% of draft length


In [ ]:
# ── CELL 10: Per-topic evaluation as a function ───────────────────────────

def save_per_topic_outputs(slug, baseline_draft, framework_raw, framework_final,
                           reference_summary=None):
    base = f"{PER_TOPIC_OUTPUTS_PATH}/{slug}"
    Path(PER_TOPIC_OUTPUTS_PATH).mkdir(parents=True, exist_ok=True)

    with open(f"{base}_baseline.txt", "w") as f:
        f.write(baseline_draft)
    with open(f"{base}_framework_raw.txt", "w") as f:
        f.write(framework_raw)
    with open(f"{base}_framework_final.txt", "w") as f:
        f.write(framework_final)
    if reference_summary is not None:
        with open(f"{base}_reference.txt", "w") as f:
            f.write(reference_summary)


def save_per_topic_graphs(slug, loop_metadata, coverage_scores,
                          factscore_scores, bertscore_scores=None):
    """
    Save five publication-grade PNGs for one topic, one file per metric:
      changes_per_loop.png  — additions kept vs ungrounded rejected, per loop
      response_sizes.png    — LLM response size per loop, colored by outcome
      semantic_coverage.png — bar chart: baseline vs framework coverage
      factscore.png         — bar chart: baseline vs framework faithfulness
      bertscore.png         — bar chart: baseline vs framework BERTScore (if reference exists)
    """
    topic_graph_dir = f"{PER_TOPIC_GRAPHS_PATH}/{slug}"
    Path(topic_graph_dir).mkdir(parents=True, exist_ok=True)

    loop_nums = sorted(loop_metadata.keys())
    additions      = [loop_metadata[n]["additions_this_loop"]            for n in loop_nums]
    rejected       = [loop_metadata[n]["rejected_ungrounded_this_loop"]  for n in loop_nums]
    outcomes       = [loop_metadata[n]["outcome"]                        for n in loop_nums]
    response_sizes = [loop_metadata[n]["raw_response_chars"]             for n in loop_nums]

    # Figure 1: per-loop changes
    fig, ax = plt.subplots(figsize=(9, 5))
    width = 0.35
    x = np.arange(len(loop_nums))
    ax.bar(x - width/2, additions, width, label="Additions kept",      color="#10B981", alpha=0.85)
    ax.bar(x + width/2, rejected,  width, label="Ungrounded rejected", color="#DC2626", alpha=0.85)
    ax.set_xlabel("Loop number")
    ax.set_ylabel("Count")
    ax.set_title(f"Per-Loop Changes — {slug}")
    ax.set_xticks(x)
    ax.set_xticklabels(loop_nums)
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(f"{topic_graph_dir}/changes_per_loop.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 2: per-loop response sizes
    outcome_colors = {"REVISED": "#4A90D9", "UNCHANGED": "#9CA3AF", "MALFORMED": "#DC2626"}
    bar_colors = [outcome_colors.get(o, "#000000") for o in outcomes]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(loop_nums, response_sizes, color=bar_colors, alpha=0.85)
    ax.set_xlabel("Loop number")
    ax.set_ylabel("Response size (chars)")
    ax.set_title(f"Per-Loop Model Response Size — {slug}")
    ax.set_xticks(loop_nums)
    legend_handles = [Patch(facecolor=v, label=k) for k, v in outcome_colors.items()]
    ax.legend(handles=legend_handles, loc="upper right")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(f"{topic_graph_dir}/response_sizes.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 3: semantic coverage
    fig, ax = plt.subplots(figsize=(7, 5))
    vals = [coverage_scores["baseline"], coverage_scores["framework"]]
    ax.bar(["Baseline", "Framework"], vals,
           color=["#9CA3AF", "#10B981"], alpha=0.85, width=0.5)
    ax.set_ylabel("Semantic Coverage")
    ax.set_title(f"Semantic Coverage — {slug}")
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3, axis="y")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{topic_graph_dir}/semantic_coverage.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 4: FActScore
    fig, ax = plt.subplots(figsize=(7, 5))
    vals = [factscore_scores["baseline"], factscore_scores["framework"]]
    ax.bar(["Baseline", "Framework"], vals,
           color=["#9CA3AF", "#4A90D9"], alpha=0.85, width=0.5)
    ax.set_ylabel("FActScore (faithfulness)")
    ax.set_title(f"FActScore — {slug}")
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3, axis="y")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{topic_graph_dir}/factscore.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Figure 5: BERTScore — only if reference summary was available
    if bertscore_scores is not None and bertscore_scores.get("baseline") is not None \
                                   and bertscore_scores.get("framework") is not None:
        fig, ax = plt.subplots(figsize=(7, 5))
        vals = [bertscore_scores["baseline"], bertscore_scores["framework"]]
        ax.bar(["Baseline", "Framework"], vals,
               color=["#9CA3AF", "#E8643A"], alpha=0.85, width=0.5)
        ax.set_ylabel("BERTScore F1 (vs reference)")
        ax.set_title(f"BERTScore — {slug}")
        ax.set_ylim(0, 1.0)
        ax.grid(alpha=0.3, axis="y")
        for i, v in enumerate(vals):
            ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{topic_graph_dir}/bertscore.png", dpi=150, bbox_inches="tight")
        plt.close()


def evaluate_topic(topic_entry):
    slug = topic_entry["slug"]
    identifier = topic_entry["identifier"]
    original_task = topic_entry["task_prompt"]

    print(f"\n{'═' * 70}")
    print(f"  EVALUATING: {slug}")
    print(f"{'═' * 70}")
    print(f"  Source            : {identifier}")
    print(f"  Prompt (preview)  : {original_task[:120]}{'...' if len(original_task) > 120 else ''}")

    print(f"\n  [1/6] Loading input...")
    input_data = load_prose(identifier)
    print(f"        Raw text: {len(input_data['raw_text']):,} chars")
    print(f"        Units: {len(input_data['units'])}")
    if input_data.get("length_bucket"):
        print(f"        Length bucket: {input_data['length_bucket']}")
    if input_data.get("reference_summary"):
        print(f"        Reference summary: {len(input_data['reference_summary']):,} chars")

    print(f"\n  [2/6] Clustering...")
    cluster_data = cluster_input(input_data["units"])
    print(f"        Clusters formed: {cluster_data['n_clusters']}")

    print(f"\n  [3/6] Generating baseline...")
    baseline = generate_baseline(input_data["raw_text"], original_task)
    print(f"        Method: {baseline['method']} | Calls: {baseline['calls']} | Time: {baseline['time']:.1f}s")
    print(f"        Draft length: {len(baseline['draft'])} chars")
    print(f"        --- BASELINE DRAFT (first 800 chars) ---")
    print(baseline["draft"][:800] + ("..." if len(baseline["draft"]) > 800 else ""))
    print(f"        --- end draft preview ---")

    print(f"\n  [4/6] Refinement loop...")
    refinement = refine(baseline["draft"], cluster_data, original_task)
    print(f"        Loops run: {refinement['n_clusters_processed']}/{refinement['n_clusters_total']}")
    print(f"        Refinement calls: {refinement['refinement_calls']} | Time: {refinement['refinement_time']:.1f}s")
    additions_kept = sum(m["additions_this_loop"] for m in refinement["loop_metadata"].values())
    rejected_ungrounded = sum(m["rejected_ungrounded_this_loop"] for m in refinement["loop_metadata"].values())
    print(f"        Additions kept: {additions_kept}")
    print(f"        Ungrounded rejected: {rejected_ungrounded}")

    print(f"\n  [5/6] Synthesis...")
    if SYNTHESIZE:
        synthesis = synthesize(baseline["draft"], refinement["all_additions"], original_task)
        if synthesis["synthesis_call_made"]:
            if synthesis.get("synthesis_skipped_reason"):
                print(f"        Synthesis ran but fell back: {synthesis['synthesis_skipped_reason']}")
            else:
                print(f"        Synthesis call made: time {synthesis['synthesis_time']:.1f}s")
        else:
            print(f"        Synthesis skipped: {synthesis.get('synthesis_skipped_reason', 'unknown')}")
    else:
        synthesis = {
            "synthesized_output": refinement["final_output_raw"],
            "synthesis_call_made": False,
            "synthesis_time": 0.0,
            "synthesis_skipped_reason": "SYNTHESIZE=False in config",
        }
        print(f"        Synthesis disabled. Using raw refined output.")
    print(f"        Synthesized length: {len(synthesis['synthesized_output'])} chars")
    print(f"        --- FINAL SYNTHESIZED OUTPUT (first 1000 chars) ---")
    print(synthesis["synthesized_output"][:1000] + ("..." if len(synthesis["synthesized_output"]) > 1000 else ""))
    print(f"        --- end final preview ---")

    print(f"\n  [6/6] Saving outputs...")
    save_per_topic_outputs(
        slug=slug,
        baseline_draft=baseline["draft"],
        framework_raw=refinement["final_output_raw"],
        framework_final=synthesis["synthesized_output"],
        reference_summary=input_data.get("reference_summary"),
    )
    print(f"        Text outputs saved to {PER_TOPIC_OUTPUTS_PATH}/{slug}_*.txt")

    framework_calls_total = baseline["calls"] + refinement["refinement_calls"] + (1 if synthesis["synthesis_call_made"] else 0)
    framework_time_total  = baseline["time"]  + refinement["refinement_time"] + synthesis["synthesis_time"]

    return {
        "slug": slug,
        "identifier": identifier,
        "task_prompt": original_task,
        "input_mode": "prose",
        "raw_text": input_data["raw_text"],
        "raw_text_chars": len(input_data["raw_text"]),
        "units_count": len(input_data["units"]),
        "length_bucket": input_data.get("length_bucket"),
        "reference_summary": input_data.get("reference_summary"),
        "n_clusters": cluster_data["n_clusters"],
        "raw_clusters_hdbscan": cluster_data.get("raw_clusters_hdbscan"),
        "noise_points": cluster_data.get("noise_points"),
        "baseline_method": baseline["method"],
        "baseline_calls": baseline["calls"],
        "baseline_time_s": baseline["time"],
        "baseline_draft": baseline["draft"],
        "refinement_calls": refinement["refinement_calls"],
        "refinement_time_s": refinement["refinement_time"],
        "loop_metadata": refinement["loop_metadata"],
        "processed_order": refinement["processed_order"],
        "additions_kept_total": additions_kept,
        "rejected_ungrounded_total": rejected_ungrounded,
        "framework_raw_output": refinement["final_output_raw"],
        "synthesis_call_made": synthesis["synthesis_call_made"],
        "synthesis_time_s": synthesis["synthesis_time"],
        "synthesis_skipped_reason": synthesis.get("synthesis_skipped_reason"),
        "final_synthesized": synthesis["synthesized_output"],
        "framework_calls_total": framework_calls_total,
        "framework_time_total": framework_time_total,
        "draft_chars": len(baseline["draft"]),
        "final_synthesized_chars": len(synthesis["synthesized_output"]),
    }


def score_topic(topic_result):
    scoring = {}

    print(f"\n  Computing prose metrics for: {topic_result['slug']}")
    source_text  = topic_result["raw_text"]
    baseline     = topic_result["baseline_draft"]
    framework    = topic_result["final_synthesized"]
    reference    = topic_result.get("reference_summary")

    print(f"    Computing FActScore (baseline)...")
    fs_b = compute_factscore(baseline, source_text)
    scoring["factscore_baseline"] = fs_b
    print(f"    Computing FActScore (framework)...")
    fs_f = compute_factscore(framework, source_text)
    scoring["factscore_framework"] = fs_f

    baseline_claims    = len(fs_b.get("claims", []))
    framework_claims   = len(fs_f.get("claims", []))
    baseline_density   = (baseline_claims  / max(len(baseline), 1))  * 1000
    framework_density  = (framework_claims / max(len(framework), 1)) * 1000
    scoring["claim_density_baseline"]  = {"claims": baseline_claims,  "per_1000_chars": baseline_density}
    scoring["claim_density_framework"] = {"claims": framework_claims, "per_1000_chars": framework_density}

    print(f"    Computing Semantic Coverage (baseline)...")
    cov_b = compute_semantic_coverage(baseline, topic_result["raw_text"])
    scoring["coverage_baseline"] = cov_b
    print(f"    Computing Semantic Coverage (framework)...")
    cov_f = compute_semantic_coverage(framework, topic_result["raw_text"])
    scoring["coverage_framework"] = cov_f

    if reference is not None and reference.strip():
        print(f"    Computing BERTScore vs reference (baseline)...")
        bs_b = compute_bertscore(baseline, reference)
        scoring["bertscore_baseline"] = bs_b
        print(f"    Computing BERTScore vs reference (framework)...")
        bs_f = compute_bertscore(framework, reference)
        scoring["bertscore_framework"] = bs_f
    else:
        scoring["bertscore_baseline"]  = {"score": None, "skipped_reason": "no_reference"}
        scoring["bertscore_framework"] = {"score": None, "skipped_reason": "no_reference"}

    return scoring


print("Cell 10: per-topic evaluation function defined")
print(f"  evaluate_topic(topic_entry)  -> result dict")
print(f"  score_topic(topic_result)    -> scoring dict (FActScore + density + Coverage + BERTScore)")
print(f"  save_per_topic_outputs(...)  -> saves _baseline.txt, _framework_raw.txt, _framework_final.txt, _reference.txt")
print(f"  save_per_topic_graphs(...)   -> 5 PNGs per topic in /graphs/per_topic/<slug>/")

Cell 10: per-topic evaluation function defined
  evaluate_topic(topic_entry)  -> result dict
  score_topic(topic_result)    -> scoring dict (FActScore + density + Coverage + BERTScore)
  save_per_topic_outputs(...)  -> saves _baseline.txt, _framework_raw.txt, _framework_final.txt, _reference.txt
  save_per_topic_graphs(...)   -> 5 PNGs per topic in /graphs/per_topic/<slug>/


In [ ]:
# ── CELL 11: FActScore, Coverage, and BERTScore as functions ──────────────

EVAL_LLM_CALLS_COUNTER = {"decomposition": 0, "verification": 0, "total": 0}


def reset_eval_counter():
    EVAL_LLM_CALLS_COUNTER["decomposition"] = 0
    EVAL_LLM_CALLS_COUNTER["verification"] = 0
    EVAL_LLM_CALLS_COUNTER["total"] = 0


def decompose_into_atomic_claims(text):
    """
    Decompose text into atomic factual claims following the FActScore paper's
    convention: each claim conveys a single, self-contained fact. The previous
    prompt produced 5-11 claims from 2-3K-char drafts because the model
    interpreted 'atomic' loosely. This prompt forces sub-clause-level granularity
    via concrete examples and explicit guidance.
    """
    if not text or not text.strip():
        return []

    prompt = f"""Decompose the text below into a complete list of atomic factual claims.

An atomic claim states ONE single fact. If a sentence contains multiple facts joined by 'and', 'while', 'because', commas, or relative clauses, you must split it into multiple atomic claims. Do not paraphrase loosely; preserve the exact factual content.

Examples of correct atomic decomposition:

Source sentence: "The 2020 report, written by Dr. Smith at the EPA, found that lead exposure declined by 40% in urban areas since 2010."
Atomic claims:
The 2020 report was written by Dr. Smith.
Dr. Smith works at the EPA.
The report found that lead exposure declined.
The decline in lead exposure was 40%.
The decline occurred in urban areas.
The decline occurred since 2010.

Source sentence: "Renewable energy production rose sharply between 2015 and 2022, driven by falling solar costs and federal tax credits."
Atomic claims:
Renewable energy production rose.
The rise was sharp.
The rise occurred between 2015 and 2022.
Falling solar costs drove the rise.
Federal tax credits drove the rise.

Rules:
- Output one claim per line.
- No numbering, no bullets, no headings, no commentary.
- Each claim must be a complete simple sentence understandable on its own.
- Do not introduce facts not present in the text.
- Decompose every sentence in the text. A typical paragraph yields 10-20 claims.

TEXT TO DECOMPOSE:
{text}

ATOMIC CLAIMS:
"""
    response = llm_call(prompt)
    EVAL_LLM_CALLS_COUNTER["decomposition"] += 1
    EVAL_LLM_CALLS_COUNTER["total"] += 1

    claims = []
    for line in response.strip().splitlines():
        line = line.strip()
        line = re.sub(r"^[-•*]\s*", "", line)
        line = re.sub(r"^\d+[\.\)]\s*", "", line)
        if line and len(line) >= 10:
            claims.append(line)

    return claims


# ── Chunked verification (unchanged from prior fix) ────────────────────────
VERIFY_CHUNK_CHARS = 25000
VERIFY_CHUNK_OVERLAP = 1000


def _chunk_source_for_verification(source_text, chunk_chars=VERIFY_CHUNK_CHARS,
                                    overlap=VERIFY_CHUNK_OVERLAP):
    if len(source_text) <= chunk_chars:
        return [source_text]
    chunks = []
    pos = 0
    while pos < len(source_text):
        end = min(pos + chunk_chars, len(source_text))
        if end < len(source_text):
            last_period = source_text.rfind(". ", pos + chunk_chars - 1000, end)
            if last_period > pos:
                end = last_period + 2
        chunks.append(source_text[pos:end])
        if end >= len(source_text):
            break
        pos = end - overlap
    return chunks


def _verify_claim_against_chunk(claim, chunk_text):
    prompt = f"""Determine whether the following claim is supported by the source document.

A claim is SUPPORTED if the source document explicitly states it or directly implies it.
A claim is NOT SUPPORTED if the source document does not state or imply it, even if the claim might be true in general.

Reply with exactly one word: SUPPORTED or NOT_SUPPORTED. No other text.

SOURCE DOCUMENT:
{chunk_text}

CLAIM:
{claim}

VERDICT:"""
    response = llm_call(prompt, num_predict=20)
    EVAL_LLM_CALLS_COUNTER["verification"] += 1
    EVAL_LLM_CALLS_COUNTER["total"] += 1

    verdict = response.strip().upper()
    if "NOT_SUPPORTED" in verdict or "NOT SUPPORTED" in verdict:
        return False
    if "SUPPORTED" in verdict:
        return True
    return None


def verify_claim_against_source(claim, source_text):
    chunks = _chunk_source_for_verification(source_text)
    for chunk in chunks:
        result = _verify_claim_against_chunk(claim, chunk)
        if result is True:
            return True
    return False


def compute_factscore(text, source_text):
    """Decompose into atomic claims, verify each, return supported/total ratio."""
    if not text or not text.strip():
        return {"score": None, "supported": 0, "total": 0, "claims": [], "verdicts": [],
                "skipped_reason": "empty_text"}

    claims = decompose_into_atomic_claims(text)
    if not claims:
        return {"score": None, "supported": 0, "total": 0, "claims": [], "verdicts": [],
                "skipped_reason": "no_claims_extracted"}

    verdicts = []
    supported = 0
    for claim in claims:
        is_supported = verify_claim_against_source(claim, source_text)
        verdicts.append(is_supported)
        if is_supported:
            supported += 1

    total = len(claims)
    score = supported / total if total > 0 else 0.0
    print(f"      {supported}/{total} claims supported = {score:.3f}")

    return {"score": score, "supported": supported, "total": total,
            "claims": claims, "verdicts": verdicts, "skipped_reason": None}


def compute_semantic_coverage(text, source_text):
    """
    Cluster the source by sentence embeddings; for each cluster centroid, find
    the max cosine similarity to any sentence in `text`. Coverage = mean across
    clusters. Range [0, 1]; higher = more of the source's distinct topics are
    semantically present in the output.
    """
    if not text or not text.strip():
        return {"score": None, "skipped_reason": "empty_text"}

    src_sents = re.split(r"(?<=[.?!])\s+(?=[A-Z])", source_text)
    src_sents = [s.strip() for s in src_sents if len(s.strip()) >= 40]
    if len(src_sents) < 2:
        return {"score": None, "skipped_reason": "source_too_short"}

    out_sents = re.split(r"(?<=[.?!])\s+(?=[A-Z])", text)
    out_sents = [s.strip() for s in out_sents if len(s.strip()) >= 20]
    if not out_sents:
        return {"score": None, "skipped_reason": "output_too_short"}

    src_emb = embedding_model.encode(src_sents, convert_to_numpy=True, show_progress_bar=False)
    out_emb = embedding_model.encode(out_sents, convert_to_numpy=True, show_progress_bar=False)

    try:
        clusterer = hdbscan.HDBSCAN(min_cluster_size=3)
        labels = clusterer.fit_predict(src_emb)
    except Exception:
        labels = np.zeros(len(src_emb), dtype=int)

    if len(set(labels) - {-1}) == 0:
        labels[:] = 0

    centroids = []
    for cid in sorted(set(labels) - {-1}):
        members = src_emb[labels == cid]
        centroids.append(members.mean(axis=0))
    centroids = np.array(centroids)

    out_norm = out_emb / (np.linalg.norm(out_emb, axis=1, keepdims=True) + 1e-9)
    cen_norm = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-9)
    sims = cen_norm @ out_norm.T
    per_cluster_max = sims.max(axis=1)
    score = float(per_cluster_max.mean())

    print(f"      coverage = {score:.3f}")
    return {"score": score, "n_clusters": len(centroids), "skipped_reason": None}


def compute_bertscore(candidate_text, reference_text):
    """
    BERTScore F1 between framework/baseline output and the GovReport human-
    written reference summary. Uses roberta-large embeddings (BERTScore default).
    Reported as F1 in [0, 1]; higher = more semantic similarity to the reference.
    """
    if not candidate_text or not candidate_text.strip():
        return {"score": None, "skipped_reason": "empty_candidate"}
    if not reference_text or not reference_text.strip():
        return {"score": None, "skipped_reason": "empty_reference"}

    try:
        P, R, F1 = bert_score_compute(
            [candidate_text],
            [reference_text],
            lang="en",
            verbose=False,
            rescale_with_baseline=False,
        )
        f1 = float(F1[0])
    except Exception as e:
        return {"score": None, "skipped_reason": f"bertscore_error: {type(e).__name__}"}

    print(f"      bertscore F1 = {f1:.3f}")
    return {"score": f1, "skipped_reason": None}


print("Cell 11: scoring functions defined")
print(f"  Prose metrics : FActScore (chunked verification, sharpened decomposition)")
print(f"                  Semantic Coverage")
print(f"                  BERTScore vs GovReport reference summary")
print(f"  Verifier      : {VERIFY_CHUNK_CHARS:,}-char chunks, {VERIFY_CHUNK_OVERLAP}-char overlap")

Cell 11: scoring functions defined
  Prose metrics : FActScore (chunked verification, sharpened decomposition)
                  Semantic Coverage
                  BERTScore vs GovReport reference summary
  Code metrics  : Bug Precision + Recall + F1 (requires PLANTED_BUGS)
  Verifier      : 25,000-char chunks, 1000-char overlap


In [ ]:
# ── CELL 11: Multi-topic orchestrator ─────────────────────────────────────
DEGENERATE_DRAFT_CHARS_THRESHOLD = 100

FORCE_RERUN_SLUGS = []


def build_topic_entries(num_topics):
    """
    Build the list of topic_entry dicts the orchestrator iterates over.
    For prose: one entry per GovReport doc in the stratified sample. The
    task_prompt is the bare instruction from GROUNDED_PROMPT_TEMPLATE; the
    document itself is loaded later by evaluate_topic via load_prose and
    fed into the baseline as raw_text. This separation prevents the document
    from being embedded in refinement or synthesis prompts where it would
    overflow the model context window on long inputs.

    Each entry has: slug, identifier, task_prompt.
    """
    sample = get_govreport_sample()
    sample = sample[:num_topics]

    entries = []
    for doc in sample:
        entries.append({
            "slug": doc["doc_id"],
            "identifier": doc["doc_id"],
            "task_prompt": GROUNDED_PROMPT_TEMPLATE,
        })
    return entries


def _load_resume_state():
    """
    Look for results_incremental.pkl in the current RUN's OUTPUT_PATH.
    Returns (prior_results, completed_slugs) where:
      - prior_results: list of result dicts to prepend to all_results
      - completed_slugs: set of slugs that are fully done (eval + scoring ok)

    A topic counts as 'fully done' only when:
      - evaluation_failed is False, AND
      - 'scoring' key exists and is a non-empty dict that does NOT have
        scoring_failed=True, AND
      - draft_chars is at least DEGENERATE_DRAFT_CHARS_THRESHOLD, AND
      - slug is not in FORCE_RERUN_SLUGS.

    Topics that are partially evaluated, hard-failed, degenerate, or
    explicitly forced are dropped so they get re-run cleanly this session.
    """
    pkl_path = f"{OUTPUT_PATH}/results_incremental.pkl"
    if not Path(pkl_path).exists():
        return [], set()

    try:
        with open(pkl_path, "rb") as f:
            prior = pickle.load(f)
    except Exception as e:
        print(f"  (warning: could not read prior pickle: {type(e).__name__}: {e})")
        print(f"  Starting from scratch.")
        return [], set()

    kept = []
    completed_slugs = set()
    dropped_partial = []
    dropped_failed = []
    dropped_degenerate = []
    dropped_forced = []

    for r in prior:
        slug = r.get("slug", "<unknown>")

        # Drop: evaluation itself hard-failed
        if r.get("evaluation_failed", False):
            dropped_failed.append(slug)
            continue

        # Drop: scoring missing or marked failed
        scoring = r.get("scoring")
        if not isinstance(scoring, dict) or not scoring or scoring.get("scoring_failed"):
            dropped_partial.append(slug)
            continue

        # Drop: baseline draft is degenerate (clearly broken output, e.g. 2-4 chars)
        draft_chars = r.get("draft_chars") or 0
        if draft_chars < DEGENERATE_DRAFT_CHARS_THRESHOLD:
            dropped_degenerate.append((slug, draft_chars))
            continue

        # Drop: explicitly listed for forced re-run
        if slug in FORCE_RERUN_SLUGS:
            dropped_forced.append(slug)
            continue

        kept.append(r)
        completed_slugs.add(slug)

    print(f"\n  RESUME: found {pkl_path}")
    print(f"    Fully completed topics (will skip)             : {len(kept)}")
    if dropped_partial:
        print(f"    Partial/unscored topics (will re-run)          : {len(dropped_partial)}")
        for s in dropped_partial:
            print(f"        - {s}")
    if dropped_failed:
        print(f"    Previously failed topics (will re-try)         : {len(dropped_failed)}")
        for s in dropped_failed:
            print(f"        - {s}")
    if dropped_degenerate:
        print(f"    Degenerate baseline (<{DEGENERATE_DRAFT_CHARS_THRESHOLD} chars) (will re-run): {len(dropped_degenerate)}")
        for s, n in dropped_degenerate:
            print(f"        - {s}  (draft_chars={n})")
    if dropped_forced:
        print(f"    Explicitly forced for re-run                   : {len(dropped_forced)}")
        for s in dropped_forced:
            print(f"        - {s}")

    return kept, completed_slugs


def _save_incremental_summary(all_results):
    """
    Write a current summary.csv after every topic so that if the session dies
    before final aggregate, the latest table is still on disk. Wrapped in
    try/except so a save error never kills the run.
    """
    try:
        df = build_summary_dataframe(all_results)
        df.to_csv(f"{OUTPUT_PATH}/summary.csv", index=False)
    except Exception as e:
        print(f"  (warning: incremental summary save failed: {type(e).__name__}: {e})")


def run_multi_topic_evaluation(num_topics=None):
    if num_topics is None:
        num_topics = EVAL_NUM_TOPICS

    num_topics = min(num_topics, STRATIFIED_SAMPLE_SIZE)

    topics_to_evaluate = build_topic_entries(num_topics)

    # Resume from any prior incremental pickle in this RUN's OUTPUT_PATH.
    # Returns previously-completed result dicts and the set of done slugs.
    prior_results, completed_slugs = _load_resume_state()

    print(f"\n{'█' * 70}")
    print(f"  MULTI-TOPIC EVALUATION RUN")
    print(f"{'█' * 70}")
    print(f"  Run tag         : {RUN_TAG}")
    print(f"  Input mode      : prose")
    print(f"  Dataset         : {DATASET_NAME}")
    print(f"  Topics to run   : {len(topics_to_evaluate)} of {STRATIFIED_SAMPLE_SIZE} in sample")
    print(f"  Pipeline model  : {PIPELINE_MODEL}")
    print(f"  Synthesis       : {'enabled' if SYNTHESIZE else 'disabled'}")
    if completed_slugs:
        remaining = len(topics_to_evaluate) - len(completed_slugs)
        print(f"  Resuming        : {len(completed_slugs)} done, {remaining} remaining this session")
    print(f"{'█' * 70}")

    reset_eval_counter()

    # Start with anything we resumed from the pickle.
    all_results = list(prior_results)
    overall_t0 = time.time()

    for idx, topic_entry in enumerate(topics_to_evaluate):
        slug = topic_entry["slug"]

        if slug in completed_slugs:
            print(f"\n  TOPIC {idx + 1} of {len(topics_to_evaluate)}: {slug}  [SKIP — already done]")
            continue

        print(f"\n\n{'▓' * 70}")
        print(f"  TOPIC {idx + 1} of {len(topics_to_evaluate)}")
        print(f"{'▓' * 70}")

        topic_t0 = time.time()

        try:
            topic_result = evaluate_topic(topic_entry)
        except Exception as e:
            print(f"\n  ✗ EVALUATION FAILED for '{slug}'")
            print(f"    Error: {type(e).__name__}: {e}")
            print(f"    Skipping this topic and continuing with the next one.")
            all_results.append({
                "slug": slug,
                "identifier": topic_entry["identifier"],
                "input_mode": "prose",
                "evaluation_failed": True,
                "error_type": type(e).__name__,
                "error_message": str(e),
            })
            try:
                with open(f"{OUTPUT_PATH}/results_incremental.pkl", "wb") as f:
                    pickle.dump(all_results, f)
            except Exception as save_e:
                print(f"  (warning: incremental save failed: {type(save_e).__name__}: {save_e})")
            _save_incremental_summary(all_results)
            continue

        try:
            scoring_result = score_topic(topic_result)
        except Exception as e:
            print(f"\n  ✗ SCORING FAILED for '{slug}'")
            print(f"    Error: {type(e).__name__}: {e}")
            print(f"    Topic evaluation kept; scoring marked as failed.")
            scoring_result = {"scoring_failed": True, "error_message": str(e)}

        if "factscore_baseline" in scoring_result:
            try:
                save_per_topic_graphs(
                    slug=slug,
                    loop_metadata=topic_result["loop_metadata"],
                    coverage_scores={
                        "baseline":  scoring_result["coverage_baseline"]["score"]  or 0.0,
                        "framework": scoring_result["coverage_framework"]["score"] or 0.0,
                    },
                    factscore_scores={
                        "baseline":  scoring_result["factscore_baseline"]["score"]  or 0.0,
                        "framework": scoring_result["factscore_framework"]["score"] or 0.0,
                    },
                    bertscore_scores={
                        "baseline":  scoring_result.get("bertscore_baseline",  {}).get("score"),
                        "framework": scoring_result.get("bertscore_framework", {}).get("score"),
                    },
                )
                print(f"\n  Per-topic graphs saved to {PER_TOPIC_GRAPHS_PATH}/{slug}/")
            except Exception as e:
                print(f"\n  ✗ Per-topic graphing failed: {type(e).__name__}: {e}")

        topic_elapsed = time.time() - topic_t0

        topic_result["scoring"] = scoring_result
        topic_result["topic_total_time"] = topic_elapsed
        topic_result["evaluation_failed"] = False

        all_results.append(topic_result)

        try:
            with open(f"{OUTPUT_PATH}/results_incremental.pkl", "wb") as f:
                pickle.dump(all_results, f)
        except Exception as e:
            print(f"  (warning: incremental save failed: {type(e).__name__}: {e})")

        _save_incremental_summary(all_results)

        print(f"\n  ✓ TOPIC {idx + 1} COMPLETE in {topic_elapsed:.1f}s")

    overall_elapsed = time.time() - overall_t0

    successful = [r for r in all_results if not r.get("evaluation_failed", False)]
    failed     = [r for r in all_results if r.get("evaluation_failed", False)]

    print(f"\n\n{'█' * 70}")
    print(f"  MULTI-TOPIC RUN COMPLETE")
    print(f"{'█' * 70}")
    print(f"  Session wall-clock time : {overall_elapsed:.1f}s ({overall_elapsed/60:.1f} min)")
    print(f"  (Resumed topics not included in session time)")
    print(f"  Topics succeeded        : {len(successful)} / {len(topics_to_evaluate)}")
    print(f"  Topics failed           : {len(failed)}")

    if successful:
        total_framework_calls = sum(r.get("framework_calls_total", 0) for r in successful)
        total_framework_time  = sum(r.get("framework_time_total", 0)  for r in successful)
        print(f"\n  FRAMEWORK COSTS (across {len(successful)} successful topics, includes resumed):")
        print(f"    Total framework calls  : {total_framework_calls}")
        print(f"    Total framework time   : {total_framework_time:.1f}s")
        print(f"    Avg per topic          : {total_framework_calls/len(successful):.1f} calls, "
              f"{total_framework_time/len(successful):.1f}s")

    print(f"\n  EVAL COSTS (this session only, separate from framework):")
    print(f"    Decomposition calls    : {EVAL_LLM_CALLS_COUNTER['decomposition']}")
    print(f"    Verification calls     : {EVAL_LLM_CALLS_COUNTER['verification']}")
    print(f"    Total eval calls       : {EVAL_LLM_CALLS_COUNTER['total']}")

    if failed:
        print(f"\n  FAILED TOPICS:")
        for r in failed:
            print(f"    {r['slug']}: {r.get('error_type', 'unknown')} — {r.get('error_message', '')[:100]}")

    print(f"\n  Results pickled to {OUTPUT_PATH}/results_incremental.pkl")
    print(f"  Latest summary CSV    : {OUTPUT_PATH}/summary.csv")
    return all_results


print("Cell 11: multi-topic orchestrator defined (with resume + incremental summary)")
print(f"  Run: results = run_multi_topic_evaluation()")
print(f"  summary.csv is refreshed after EVERY topic (not just at the end)")
print(f"  task_prompt now carries the bare instruction (no document embedding)")

Cell 11: multi-topic orchestrator defined (with resume + incremental summary)
  Run: results = run_multi_topic_evaluation()
  summary.csv is refreshed after EVERY topic (not just at the end)
  task_prompt now carries the bare instruction (no document embedding)


In [ ]:
# ── CELL 13: Aggregate graphs and summary table ───────────────────────────

def _safe_score(scoring_dict, key):
    """Extract a 0-1 score from a scoring sub-dict, returning None when skipped."""
    sub = scoring_dict.get(key)
    if not isinstance(sub, dict):
        return None
    return sub.get("score")


def build_summary_dataframe(all_results):
    rows = []
    for r in all_results:
        if r.get("evaluation_failed", False):
            rows.append({
                "slug": r["slug"],
                "identifier": r.get("identifier", ""),
                "evaluation_failed": True,
                "error": f"{r.get('error_type', '')}: {r.get('error_message', '')[:80]}",
            })
            continue

        scoring = r.get("scoring", {})
        row = {
            "slug": r["slug"],
            "identifier": r.get("identifier", ""),
            "length_bucket": r.get("length_bucket", ""),
            "evaluation_failed": False,
            "raw_text_chars": r.get("raw_text_chars", 0),
            "units_count": r.get("units_count", 0),
            "n_clusters": r.get("n_clusters", 0),
            "raw_clusters_hdbscan": r.get("raw_clusters_hdbscan", 0),
            "noise_points": r.get("noise_points", 0),
            "baseline_method": r.get("baseline_method", ""),
            "baseline_calls": r.get("baseline_calls", 0),
            "baseline_time_s": round(r.get("baseline_time_s", 0), 1),
            "refinement_calls": r.get("refinement_calls", 0),
            "refinement_time_s": round(r.get("refinement_time_s", 0), 1),
            "framework_calls_total": r.get("framework_calls_total", 0),
            "framework_time_s": round(r.get("framework_time_total", 0), 1),
            "additions_kept": r.get("additions_kept_total", 0),
            "ungrounded_rejected": r.get("rejected_ungrounded_total", 0),
            "draft_chars": r.get("draft_chars", 0),
            "final_synthesized_chars": r.get("final_synthesized_chars", 0),
        }

        fs_b  = _safe_score(scoring, "factscore_baseline")
        fs_f  = _safe_score(scoring, "factscore_framework")
        cov_b = _safe_score(scoring, "coverage_baseline")
        cov_f = _safe_score(scoring, "coverage_framework")
        bs_b  = _safe_score(scoring, "bertscore_baseline")
        bs_f  = _safe_score(scoring, "bertscore_framework")

        cd_b = scoring.get("claim_density_baseline",  {}) or {}
        cd_f = scoring.get("claim_density_framework", {}) or {}

        row["factscore_baseline"]  = round(fs_b, 4) if fs_b is not None else None
        row["factscore_framework"] = round(fs_f, 4) if fs_f is not None else None
        row["factscore_delta"]     = round(fs_f - fs_b, 4) if (fs_b is not None and fs_f is not None) else None
        row["coverage_baseline"]   = round(cov_b, 4) if cov_b is not None else None
        row["coverage_framework"]  = round(cov_f, 4) if cov_f is not None else None
        row["coverage_delta"]      = round(cov_f - cov_b, 4) if (cov_b is not None and cov_f is not None) else None
        row["bertscore_baseline"]  = round(bs_b, 4) if bs_b is not None else None
        row["bertscore_framework"] = round(bs_f, 4) if bs_f is not None else None
        row["bertscore_delta"]     = round(bs_f - bs_b, 4) if (bs_b is not None and bs_f is not None) else None
        row["claims_baseline"]     = cd_b.get("claims")
        row["claims_framework"]    = cd_f.get("claims")
        row["claim_density_baseline_per1k"]  = round(cd_b.get("per_1000_chars", 0), 3) if cd_b else None
        row["claim_density_framework_per1k"] = round(cd_f.get("per_1000_chars", 0), 3) if cd_f else None

        rows.append(row)

    return pd.DataFrame(rows)


def compute_significance_tests(df):
    """
    Wilcoxon signed-rank test on paired (baseline, framework) scores.
    Returns a dict of {metric: {n, mean_delta, statistic, p_value, significant_05}}.
    Wilcoxon is the right test here because:
      - Pairs come from the same documents (paired design).
      - Sample is small (n=30) and not assumed normal.
      - Non-parametric -> robust to outlier topics.
    """
    successful = df[df["evaluation_failed"] == False]
    tests = {}

    metric_pairs = [
        ("factscore",  "factscore_baseline",  "factscore_framework"),
        ("coverage",   "coverage_baseline",   "coverage_framework"),
        ("bertscore",  "bertscore_baseline",  "bertscore_framework"),
    ]

    for name, col_b, col_f in metric_pairs:
        if col_b not in successful.columns or col_f not in successful.columns:
            continue
        paired = successful[[col_b, col_f]].dropna()
        n = len(paired)
        if n < 5:
            tests[name] = {"n": n, "skipped_reason": f"only_{n}_pairs_need_5+"}
            continue

        diffs = paired[col_f].values - paired[col_b].values
        mean_delta = float(diffs.mean())

        if np.allclose(diffs, 0):
            tests[name] = {"n": n, "mean_delta": 0.0, "statistic": None,
                           "p_value": 1.0, "significant_05": False,
                           "note": "no_differences_between_pairs"}
            continue

        try:
            stat, p_value = scipy_stats.wilcoxon(diffs, zero_method="wilcox", alternative="two-sided")
            tests[name] = {
                "n": n,
                "mean_delta": round(mean_delta, 4),
                "statistic": round(float(stat), 4),
                "p_value": round(float(p_value), 6),
                "significant_05": bool(p_value < 0.05),
            }
        except Exception as e:
            tests[name] = {"n": n, "mean_delta": round(mean_delta, 4),
                           "skipped_reason": f"{type(e).__name__}: {e}"}

    return tests


# ── Six aggregate figures (unchanged plotting code, only the title text and
# input-key updates where keys changed) ──────────────────────────────────

def plot_aggregate_cost_comparison(df, save_path):
    successful = df[df["evaluation_failed"] == False]
    if successful.empty:
        return
    baseline_calls = successful["baseline_calls"].values
    framework_calls = successful["framework_calls_total"].values

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(2)
    means = [baseline_calls.mean(), framework_calls.mean()]
    stds  = [baseline_calls.std(),  framework_calls.std()]
    ax.bar(x, means, yerr=stds, capsize=8,
           color=["#9CA3AF", "#4A90D9"], alpha=0.85, width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(["Baseline", "Framework"])
    ax.set_ylabel("LLM calls per topic")
    ax.set_title(f"Cost Comparison (mean ± std across {len(successful)} topics)")
    ax.grid(alpha=0.3, axis="y")
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.5, f"{m:.1f}±{s:.1f}", ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_aggregate_factscore_distribution(df, save_path):
    successful = df[(df["evaluation_failed"] == False) & df["factscore_baseline"].notna()]
    if successful.empty:
        return
    data = pd.DataFrame({
        "Baseline":  successful["factscore_baseline"].values,
        "Framework": successful["factscore_framework"].values,
    })
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.violinplot(data=data, ax=ax, palette=["#9CA3AF", "#4A90D9"], inner="quartile")
    sns.stripplot(data=data, ax=ax, color="black", alpha=0.5, size=4, jitter=0.05)
    ax.set_ylabel("FActScore (faithfulness)")
    ax.set_title(f"Faithfulness Distribution Across {len(successful)} Topics")
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_aggregate_coverage_distribution(df, save_path):
    successful = df[(df["evaluation_failed"] == False) & df["coverage_baseline"].notna()]
    if successful.empty:
        return
    data = pd.DataFrame({
        "Baseline":  successful["coverage_baseline"].values,
        "Framework": successful["coverage_framework"].values,
    })
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.violinplot(data=data, ax=ax, palette=["#9CA3AF", "#10B981"], inner="quartile")
    sns.stripplot(data=data, ax=ax, color="black", alpha=0.5, size=4, jitter=0.05)
    ax.set_ylabel("Semantic Coverage")
    ax.set_title(f"Coverage Distribution Across {len(successful)} Topics")
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_aggregate_pruning_behavior(all_results, save_path):
    successful = [r for r in all_results
                  if not r.get("evaluation_failed", False) and "loop_metadata" in r]
    if not successful:
        return
    max_loops_observed = max(
        max(r["loop_metadata"].keys()) if r["loop_metadata"] else 0
        for r in successful
    )
    curves = []
    for r in successful:
        n_clusters = r["n_clusters"]
        loop_meta = r["loop_metadata"]
        curve = [n_clusters]
        for loop_num in sorted(loop_meta.keys()):
            curve.append(loop_meta[loop_num]["active_remaining"])
        while len(curve) < max_loops_observed + 1:
            curve.append(curve[-1])
        curves.append(curve[:max_loops_observed + 1])
    curves_arr = np.array(curves)
    median = np.median(curves_arr, axis=0)
    q25 = np.percentile(curves_arr, 25, axis=0)
    q75 = np.percentile(curves_arr, 75, axis=0)
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(median))
    ax.plot(x, median, marker="o", linewidth=2, color="#E8643A", label="Median")
    ax.fill_between(x, q25, q75, alpha=0.25, color="#E8643A", label="IQR (25-75%)")
    ax.set_xlabel("Loop number")
    ax.set_ylabel("Active clusters remaining")
    ax.set_title(f"Pruning Behavior Across {len(successful)} Topics")
    ax.set_xticks(x)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_aggregate_rejection_rate(all_results, save_path):
    successful = [r for r in all_results
                  if not r.get("evaluation_failed", False) and "loop_metadata" in r]
    if not successful:
        return
    rejection_rates = []
    for r in successful:
        kept = sum(m["additions_this_loop"] for m in r["loop_metadata"].values())
        rejected = sum(m["rejected_ungrounded_this_loop"] for m in r["loop_metadata"].values())
        total = kept + rejected
        rejection_rates.append(rejected / total if total > 0 else 0.0)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(rejection_rates, bins=10, color="#DC2626", alpha=0.75, edgecolor="black")
    ax.axvline(np.mean(rejection_rates), color="black", linestyle="--", linewidth=2,
               label=f"Mean: {np.mean(rejection_rates):.2%}")
    ax.set_xlabel("Verbatim Grounding Rejection Rate")
    ax.set_ylabel("Number of topics")
    ax.set_title(f"Rejection Rate Distribution Across {len(successful)} Topics")
    ax.set_xlim(0, 1.0)
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_aggregate_cost_vs_quality(df, save_path):
    successful = df[(df["evaluation_failed"] == False) & df["factscore_delta"].notna()]
    if successful.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(successful["framework_calls_total"], successful["factscore_delta"],
               color="#4A90D9", alpha=0.75, s=80, edgecolor="black")
    ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    for _, row in successful.iterrows():
        ax.annotate(row["slug"], (row["framework_calls_total"], row["factscore_delta"]),
                    fontsize=7, alpha=0.7, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Framework LLM calls (total per topic)")
    ax.set_ylabel("FActScore gain (framework − baseline)")
    ax.set_title(f"Cost vs Quality Trade-off Across {len(successful)} Topics")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def aggregate_and_save(all_results):
    print(f"\n{'═' * 70}")
    print(f"  AGGREGATE GRAPHS AND SUMMARY")
    print(f"{'═' * 70}")

    df = build_summary_dataframe(all_results)

    csv_path = f"{OUTPUT_PATH}/summary.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n  Summary CSV saved: {csv_path}")
    print(f"    Rows: {len(df)} | Columns: {len(df.columns)}")

    print(f"\n  Generating aggregate figures...")
    plot_aggregate_cost_comparison(df,            f"{AGGREGATE_GRAPHS_PATH}/fig1_cost_comparison.png");          print(f"    [1/6] Cost comparison")
    plot_aggregate_factscore_distribution(df,     f"{AGGREGATE_GRAPHS_PATH}/fig2_factscore_distribution.png");   print(f"    [2/6] FActScore distribution")
    plot_aggregate_coverage_distribution(df,      f"{AGGREGATE_GRAPHS_PATH}/fig3_coverage_distribution.png");    print(f"    [3/6] Coverage distribution")
    plot_aggregate_pruning_behavior(all_results,  f"{AGGREGATE_GRAPHS_PATH}/fig4_pruning_behavior.png");         print(f"    [4/6] Pruning behavior")
    plot_aggregate_rejection_rate(all_results,    f"{AGGREGATE_GRAPHS_PATH}/fig5_rejection_rate.png");           print(f"    [5/6] Verbatim grounding rejection rate")
    plot_aggregate_cost_vs_quality(df,            f"{AGGREGATE_GRAPHS_PATH}/fig6_cost_vs_quality.png");          print(f"    [6/6] Cost vs quality scatter")
    print(f"\n  All aggregate figures saved to: {AGGREGATE_GRAPHS_PATH}/")

    successful = df[df["evaluation_failed"] == False]
    if not successful.empty:
        print(f"\n{'═' * 70}")
        print(f"  RESULTS SUMMARY")
        print(f"{'═' * 70}")
        print(f"  Topics evaluated      : {len(successful)}")

        if "factscore_framework" in successful.columns:
            fs_b = successful["factscore_baseline"].mean()
            fs_f = successful["factscore_framework"].mean()
            print(f"\n  FActScore (mean):")
            print(f"    Baseline  : {fs_b:.4f}")
            print(f"    Framework : {fs_f:.4f}")
            print(f"    Delta     : {fs_f - fs_b:+.4f}")

        if "coverage_framework" in successful.columns:
            cov_b = successful["coverage_baseline"].mean()
            cov_f = successful["coverage_framework"].mean()
            print(f"\n  Semantic Coverage (mean):")
            print(f"    Baseline  : {cov_b:.4f}")
            print(f"    Framework : {cov_f:.4f}")
            print(f"    Delta     : {cov_f - cov_b:+.4f}")

        if "bertscore_framework" in successful.columns and successful["bertscore_baseline"].notna().any():
            bs_b = successful["bertscore_baseline"].mean()
            bs_f = successful["bertscore_framework"].mean()
            print(f"\n  BERTScore (vs reference, mean):")
            print(f"    Baseline  : {bs_b:.4f}")
            print(f"    Framework : {bs_f:.4f}")
            print(f"    Delta     : {bs_f - bs_b:+.4f}")

        if "claims_baseline" in successful.columns:
            cb = successful["claims_baseline"].mean()
            cf = successful["claims_framework"].mean()
            dbp = successful["claim_density_baseline_per1k"].mean()
            dfp = successful["claim_density_framework_per1k"].mean()
            print(f"\n  Claim count (mean per topic):")
            print(f"    Baseline  : {cb:.1f} claims  ({dbp:.2f} per 1000 chars)")
            print(f"    Framework : {cf:.1f} claims  ({dfp:.2f} per 1000 chars)")

        calls_b = successful["baseline_calls"].mean()
        calls_f = successful["framework_calls_total"].mean()
        print(f"\n  LLM calls (mean per topic):")
        print(f"    Baseline  : {calls_b:.1f}")
        print(f"    Framework : {calls_f:.1f}")
        print(f"    Cost ratio: {calls_f/calls_b:.2f}x")

        # ── Statistical significance ──────────────────────────────────────
        print(f"\n{'═' * 70}")
        print(f"  STATISTICAL SIGNIFICANCE (Wilcoxon signed-rank, paired)")
        print(f"{'═' * 70}")
        tests = compute_significance_tests(df)
        for metric_name, result in tests.items():
            print(f"\n  {metric_name.upper()}:")
            if "skipped_reason" in result:
                print(f"    Skipped: {result['skipped_reason']}")
                continue
            print(f"    n              : {result['n']} pairs")
            print(f"    mean delta     : {result['mean_delta']:+.4f}")
            if result.get("statistic") is not None:
                print(f"    test statistic : {result['statistic']}")
                print(f"    p-value        : {result['p_value']}")
                sig_mark = "✓ SIGNIFICANT (p<0.05)" if result["significant_05"] else "✗ not significant (p>=0.05)"
                print(f"    verdict        : {sig_mark}")
            if "note" in result:
                print(f"    note           : {result['note']}")

        # Persist significance to a small JSON next to the CSV for the paper
        sig_path = f"{OUTPUT_PATH}/significance_tests.json"
        try:
            with open(sig_path, "w") as f:
                json.dump(tests, f, indent=2)
            print(f"\n  Significance tests saved: {sig_path}")
        except Exception as e:
            print(f"  (warning: could not save significance JSON: {e})")

    return df


print("Cell 13: aggregation functions defined")
print(f"  Run: df = aggregate_and_save(results)")
print(f"  Produces: 6 aggregate figures + summary.csv + significance_tests.json + console summary")

Cell 13: aggregation functions defined
  Run: df = aggregate_and_save(results)
  Produces: 6 aggregate figures + summary.csv + significance_tests.json + console summary


In [ ]:
print('Running multi-topic evaluation... this may take a while!')
results = run_multi_topic_evaluation()

# Brief per-topic summary instead of dumping the full results dict
print("\n\n" + "=" * 70)
print("  PER-TOPIC SUMMARY")
print("=" * 70)
for r in results:
    if r.get("evaluation_failed", False):
        print(f"  [FAIL] {r['slug']}: {r.get('error_type', '')} {r.get('error_message', '')[:80]}")
        continue
    sc = r.get("scoring", {})
    fs_b = sc.get("factscore_baseline", {}).get("score") if isinstance(sc.get("factscore_baseline"), dict) else None
    fs_f = sc.get("factscore_framework", {}).get("score") if isinstance(sc.get("factscore_framework"), dict) else None
    cov_b = sc.get("coverage_baseline", {}).get("score") if isinstance(sc.get("coverage_baseline"), dict) else None
    cov_f = sc.get("coverage_framework", {}).get("score") if isinstance(sc.get("coverage_framework"), dict) else None
    print(f"  [OK]   {r['slug']}: clusters={r.get('n_clusters')}, "
          f"refine_calls={r.get('refinement_calls')}, "
          f"FActScore {fs_b:.3f}->{fs_f:.3f}" if fs_b is not None and fs_f is not None
          else f"  [OK]   {r['slug']}: clusters={r.get('n_clusters')}, refine_calls={r.get('refinement_calls')}")


Running multi-topic evaluation... this may take a while!

  RESUME: found /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/outputs/results_incremental.pkl
    Fully completed topics (will skip)             : 30

██████████████████████████████████████████████████████████████████████
  MULTI-TOPIC EVALUATION RUN
██████████████████████████████████████████████████████████████████████
  Run tag         : llama3.1-8b_synth_on
  Input mode      : prose
  Dataset         : ccdv/govreport-summarization
  Topics to run   : 30 of 30 in sample
  Pipeline model  : llama3.1:8b
  Synthesis       : enabled
  Resuming        : 30 done, 0 remaining this session
██████████████████████████████████████████████████████████████████████

  TOPIC 1 of 30: govreport_00957  [SKIP — already done]

  TOPIC 2 of 30: govreport_00727  [SKIP — already done]

  TOPIC 3 of 30: govreport_00094  [SKIP — already done]

  TOPIC 4 of 30: govreport_00626  [SKIP — already done]

  TOPIC 5 of 30: govreport_00400

In [ ]:
import pickle
from pathlib import Path

# Load results from the incremental pickle file
pkl_path = Path(f"{OUTPUT_PATH}/results_incremental.pkl")

if pkl_path.exists():
    with open(pkl_path, "rb") as f:
        results = pickle.load(f)
    print(f"Loaded 'results' from {pkl_path}")
else:
    print(f"Warning: {pkl_path} not found. 'results' variable remains undefined or incomplete.")
    # If the file doesn't exist, 'results' will remain undefined, which will lead to the same error.
    # In a real scenario, you might want to re-run run_multi_topic_evaluation() or handle this more gracefully.


Loaded 'results' from /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/outputs/results_incremental.pkl


In [ ]:
print('\nGenerating aggregate graphs and summary...')
df_summary = aggregate_and_save(results)
display(df_summary)


Generating aggregate graphs and summary...

══════════════════════════════════════════════════════════════════════
  AGGREGATE GRAPHS AND SUMMARY
══════════════════════════════════════════════════════════════════════

  Summary CSV saved: /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/outputs/summary.csv
    Rows: 30 | Columns: 33

  Generating aggregate figures...
    [1/6] Cost comparison
    [2/6] FActScore distribution
    [3/6] Coverage distribution
    [4/6] Pruning behavior
    [5/6] Verbatim grounding rejection rate
    [6/6] Cost vs quality scatter

  All aggregate figures saved to: /content/drive/MyDrive/thesis_pipeline/runs/llama3.1-8b_synth_on/outputs/graphs/aggregate/

══════════════════════════════════════════════════════════════════════
  RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════
  Topics evaluated      : 30

  FActScore (mean):
    Baseline  : 0.9724
    Framework : 0.9828
    Delta     : +0.0104

  Semanti

,slug,identifier,length_bucket,evaluation_failed,raw_text_chars,units_count,n_clusters,raw_clusters_hdbscan,noise_points,baseline_method,...,coverage_baseline,coverage_framework,coverage_delta,bertscore_baseline,bertscore_framework,bertscore_delta,claims_baseline,claims_framework,claim_density_baseline_per1k,claim_density_framework_per1k
0,govreport_00957,govreport_00957,short,False,19129,115,8,11,22,single_pass,...,0.7595,0.7752,0.0157,0.8564,0.8657,0.0093,46,49,15.567,10.479
1,govreport_00727,govreport_00727,short,False,21429,128,8,11,27,map_reduce,...,0.7400,0.8389,0.0989,0.8374,0.8354,-0.0020,77,34,46.752,8.311
2,govreport_00094,govreport_00094,short,False,29797,178,8,9,13,map_reduce,...,0.7734,0.7849,0.0115,0.8329,0.8621,0.0292,51,52,19.340,12.440
3,govreport_00626,govreport_00626,short,False,29953,142,8,15,33,map_reduce,...,0.7834,0.8554,0.0720,0.8493,0.8584,0.0091,65,87,17.119,8.844
4,govreport_00400,govreport_00400,short,False,28171,137,8,15,32,map_reduce,...,0.7629,0.7716,0.0087,0.8462,0.8532,0.0070,27,64,11.269,13.485
5,govreport_00393,govreport_00393,short,False,24254,143,8,15,38,map_reduce,...,0.6732,0.7364,0.0632,0.8578,0.8575,-0.0002,31,52,17.940,12.378
6,govreport_00689,govreport_00689,short,False,18292,96,5,5,17,single_pass,...,0.6947,0.8538,0.1591,0.8898,0.8927,0.0029,36,36,11.635,6.301
7,govreport_00117,govreport_00117,short,False,17570,97,8,12,27,single_pass,...,0.6960,0.8484,0.1524,0.8683,0.8702,0.0019,43,58,22.513,12.773
8,govreport_00195,govreport_00195,short,False,18755,106,3,2,1,single_pass,...,0.8211,0.8211,0.0000,0.8540,0.8545,0.0005,30,25,11.240,7.321
9,govreport_00826,govreport_00826,short,False,29036,113,8,16,19,map_reduce,...,0.8495,0.8132,-0.0363,0.8423,0.8535,0.0112,18,37,6.608,9.444
